
# Logistic Regression & Forward Selection — **X* (18 features)** + Undersampling

Order of analysis:
1. **Logistic regression with non-standardized features** — coefficient plot  
2. **Logistic regression with standardized features** — coefficient plot  
3. **Forward selection using standardized features** (AIC/BIC tracked)  
4. **Forward selection using standardized features with UNDERSAMPLING of the majority class**

**Feature restriction**: only numeric columns starting with `X`. If more than 18 exist, use the first 18 after sorting by name.  
**Target**: `status_label`, where `alive → 0` and `failed → 1`.  
**Loss**: unregularized logistic regression (penalty `None`), i.e., MLE minimizing log loss (negative log-likelihood).


In [1]:
!git clone https://github.com/AlexandreAronne/Logistic-Regression-Interpretability.git

'git' não é reconhecido como um comando interno
ou externo, um programa operável ou um arquivo em lotes.


In [2]:

# === Imports & Config ===
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, log_loss
from sklearn.model_selection import train_test_split
from sklearn.utils import resample

RANDOM_STATE = 42
TEST_SIZE = 0.2

# ==== EDIT THIS PATH ====
CSV_PATH = "Logistic-Regression-Interpretability/american_bankruptcy_dataset_new_light.xlsx"  # <-- change to your local CSV path
TARGET_COL = "status_label"
X_PREFIX = "X"
X_TARGET_COUNT = 18


# Optional hover support
try:
    import mplcursors  # for hover tooltips on coefficient plots
except Exception:
    mplcursors = None


In [3]:

# === Load & Prepare Data (x-only) ===
df = pd.read_excel(CSV_PATH)

# Map target if object
if df[TARGET_COL].dtype == 'object':
    ser = df[TARGET_COL].str.lower().str.strip()
    df[TARGET_COL] = ser.map({'alive': 0, 'failed': 1}).astype(int)

# Keep only numeric 'x*' columns
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
x_cols = sorted([c for c in num_cols if c.startswith(X_PREFIX) and c != TARGET_COL])

# Enforce exactly 18 when possible
if len(x_cols) >= X_TARGET_COUNT:
    x_cols = x_cols[:X_TARGET_COUNT]
    print(f"Using the first {X_TARGET_COUNT} 'x*' features (sorted):", x_cols)
else:
    print(f"WARNING: Found only {len(x_cols)} 'x*' numeric features. Using all:", x_cols)

# Features and target
X_full = df[x_cols].copy()
y_full = df[TARGET_COL].astype(int)

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_full,
    y_full,
    test_size=TEST_SIZE,      # already defined as 0.2
    random_state=RANDOM_STATE,
    stratify=y_full           # keep class balance in split
)

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)
print("Train class balance:", y_train.value_counts().to_dict())
print("Test class balance:",  y_test.value_counts().to_dict())


FileNotFoundError: [Errno 2] No such file or directory: 'Logistic-Regression-Interpretability/american_bankruptcy_dataset_new_light.xlsx'

In [ ]:

# === Helper Functions ===

def logistic_mle(X, y):
    """Fit unregularized logistic regression and return model and predicted probabilities on X."""
    model = LogisticRegression(penalty=None, solver='lbfgs', max_iter=5000, random_state=RANDOM_STATE, class_weight='balanced')
    model.fit(X, y)
    p = model.predict_proba(X)[:, 1]
    return model, p

def log_likelihood_binom(y, p, eps=1e-12):
    p = np.clip(p, eps, 1 - eps)
    return np.sum(y * np.log(p) + (1 - y) * np.log(1 - p))

def aic_bic_from_model(X, y, model, p_hat):
    n = X.shape[0]
    k = X.shape[1] + 1  # coefficients + intercept
    ll = log_likelihood_binom(y, p_hat)
    aic = 2 * k - 2 * ll
    bic = np.log(n) * k - 2 * ll
    return aic, bic, ll

def plot_coefficients(coef, feature_names, title):
    plt.figure(figsize=(10, 4))
    idx = np.argsort(coef)
    plt.bar(range(len(coef)), coef[idx])
    plt.xticks(range(len(coef)), np.array(feature_names)[idx], rotation=90)
    plt.title(title)
    plt.tight_layout()
    plt.show()

def forward_selection(X, y, feature_names, criterion='AIC', max_steps=None, verbose=True):
    if criterion not in {'AIC', 'BIC'}:
        raise ValueError("criterion must be 'AIC' or 'BIC'")
    remaining = list(range(X.shape[1]))
    selected = []
    best_score = np.inf
    history = {
        'step': [], 'selected_feature': [], 'num_features': [],
        'AIC': [], 'BIC': [], 'loglik': [],
        'accuracy': [], 'precision': [], 'recall': [], 'f1': [], 'roc_auc': []
    }
    step = 0
    while remaining:
        candidates = []
        for j in remaining:
            cols = selected + [j]
            X_sub = X[:, cols]
            try:
                model, p_hat = logistic_mle(X_sub, y)
                aic, bic, ll = aic_bic_from_model(X_sub, y, model, p_hat)
                score = aic if criterion == 'AIC' else bic
                y_pred = (p_hat >= 0.5).astype(int)
                metrics = dict(
                    accuracy=accuracy_score(y, y_pred),
                    precision=precision_score(y, y_pred, zero_division=0),
                    recall=recall_score(y, y_pred, zero_division=0),
                    f1=f1_score(y, y_pred, zero_division=0),
                    roc_auc=roc_auc_score(y, p_hat) if len(np.unique(y)) == 2 else np.nan
                )
                candidates.append((score, aic, bic, ll, j, metrics))
            except Exception:
                continue
        if not candidates:
            if verbose: print("No viable candidates remain. Stopping.")
            break
        candidates.sort(key=lambda t: t[0])
        score, aic, bic, ll, best_j, metrics = candidates[0]
        if score >= best_score - 1e-9:
            if verbose: print(f"No improvement in {criterion}. Stopping at step {step}.")
            break
        selected.append(best_j); remaining.remove(best_j); best_score = score; step += 1
        history['step'].append(step)
        history['selected_feature'].append(feature_names[best_j])
        history['num_features'].append(len(selected))
        history['AIC'].append(aic); history['BIC'].append(bic); history['loglik'].append(ll)
        history['accuracy'].append(metrics['accuracy'])
        history['precision'].append(metrics['precision'])
        history['recall'].append(metrics['recall'])
        history['f1'].append(metrics['f1'])
        history['roc_auc'].append(metrics['roc_auc'])
        if verbose:
            print(f"Step {step}: added '{feature_names[best_j]}' | AIC={aic:.2f}, BIC={bic:.2f}")
        if max_steps is not None and step >= max_steps:
            if verbose: print("Reached max_steps limit.")
            break
    return selected, pd.DataFrame(history)

def plot_history_metrics(history_df, title_prefix="Forward Selection"):
    if history_df.empty:
        print("History is empty — nothing to plot."); return
    x = history_df['num_features']
    for metric in ['AIC', 'BIC', 'f1', 'accuracy', 'precision', 'recall', 'roc_auc']:
        plt.figure(figsize=(6,4))
        plt.plot(x, history_df[metric], marker='o')
        plt.xlabel("Number of selected features"); plt.ylabel(metric)
        plt.title(f"{title_prefix}: {metric} vs. #features")
        plt.grid(True, linestyle='--', alpha=0.5)
        plt.tight_layout(); plt.show()


def plot_coefficients(coef, feature_names, title, height_per_feature=0.5, show_odds_ratio=True):
    """Horizontal coefficient bar plot with symmetric x-limits and optional odds-ratio axis.

    - Bars are sorted by absolute value (descending)
    - Features on the y-axis, coefficients on the x-axis
    - Vertical zero line
    - Optional top secondary x-axis mapping coef -> exp(coef)
    - Hover tooltips if `mplcursors` is available
    """
    coef = np.asarray(coef).ravel()
    feature_names = np.asarray(feature_names)
    # Drop NaNs if any
    mask = ~np.isnan(coef)
    coef = coef[mask]
    feature_names = feature_names[mask]

    order = np.argsort(-np.abs(coef))
    coef = coef[order]
    feature_names = feature_names[order]

    n = len(coef)
    fig_h = max(4.0, height_per_feature * n)
    fig, ax = plt.subplots(figsize=(10, fig_h))
    bars = ax.barh(range(n), coef, align='center')
    ax.set_yticks(range(n))
    ax.set_yticklabels(feature_names)
    ax.invert_yaxis()  # largest at top
    ax.set_xlabel('Coefficient')
    ax.set_title(title)

    # Symmetric limits around 0
    max_abs = float(np.max(np.abs(coef))) if n > 0 else 1.0
    lim = max_abs * 1.10 if max_abs > 0 else 1.0
    ax.set_xlim(-lim, lim)
    ax.axvline(0.0, linestyle='--', linewidth=1)

    # Optional odds-ratio axis
    if show_odds_ratio:
        def fwd(x): 
            return np.exp(x)
        def inv(x): 
            return np.log(x)
        try:
            ax_top = ax.secondary_xaxis('top', functions=(fwd, inv))
            ax_top.set_xlabel('Odds ratio (exp(coef))')
        except Exception:
            pass  # secondary axis requires Matplotlib >= 3.1

    plt.tight_layout()

    # Hover tooltips with mplcursors, if available
    if 'mplcursors' in globals() and mplcursors is not None:
        cursor = mplcursors.cursor(bars, hover=True)
        @cursor.connect("add")
        def on_add(sel):
            idx = int(sel.target.index)
            sel.annotation.set(text=f"{feature_names[idx]}: {coef[idx]:.6f}")

    plt.show()



## Class Distribution by Year (Stacked)

This plot shows the number of **Solvent (0)** and **Insolvent (1)** firms per year (auto-detected year column: `fyear` → `year`/`Year` → `FYEAR` → first datetime column).

In [ ]:
# === Class Distribution by Year (Stacked) ===
# Auto-detect year column
year_candidates = ['fyear', 'year', 'Year', 'FYEAR']
year_col = None

for cand in year_candidates:
    if cand in df.columns:
        year_col = cand
        break

if year_col is None:
    # look for first datetime column
    dt_cols = [c for c in df.columns if np.issubdtype(df[c].dtype, np.datetime64)]
    if dt_cols:
        year_col = dt_cols[0]
        df['_year_tmp'] = pd.to_datetime(df[year_col]).dt.year
        year_col = '_year_tmp'

if year_col is not None:
    # Prepare counts by class
    tmp = df[[year_col, TARGET_COL]].copy()
    if np.issubdtype(tmp[year_col].dtype, np.datetime64):
        tmp[year_col] = pd.to_datetime(tmp[year_col]).dt.year
    tmp['__year__'] = tmp[year_col].astype(int)
    ct = tmp.groupby(['__year__', df[TARGET_COL].astype(int)]).size().unstack(fill_value=0)
    # ensure both classes 0 and 1 exist as columns
    for cls in [0,1]:
        if cls not in ct.columns:
            ct[cls] = 0
    ct = ct.sort_index()

    ax = ct[[0,1]].plot(kind='bar', stacked=True)
    ax.set_xlabel('Year')
    ax.set_ylabel('Count of firms')
    ax.legend(['Solvent (0)', 'Insolvent (1)'])
    ax.set_title('Class Distribution by Year (Stacked)')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

    # Clean up temp column if used
    if '_year_tmp' in df.columns:
        df.drop(columns=['_year_tmp'], inplace=True)
else:
    print("No year-like column found (checked: fyear, year/Year, FYEAR, and datetime columns). Skipping year distribution plot.")

## 1) Logistic Regression with **Non-Standardized** Features — Coefficient Plot

In [ ]:

# With this:
RANDOM_STATE = 42
model_raw = LogisticRegression(penalty=None, solver='lbfgs', max_iter=5000, random_state=RANDOM_STATE, class_weight='balanced')
model_raw.fit(X_train, y_train)

p_test = model_raw.predict_proba(X_test)[:, 1]
y_pred = (p_test >= 0.5).astype(int)

print("Test metrics (raw features):")
print({
    'accuracy': accuracy_score(y_test, y_pred),
    'precision': precision_score(y_test, y_pred, zero_division=0),
    'recall': recall_score(y_test, y_pred, zero_division=0),
    'f1': f1_score(y_test, y_pred, zero_division=0),
    'roc_auc': roc_auc_score(y_test, p_test) if len(np.unique(y_test)) == 2 else np.nan,
    'log_loss': log_loss(y_test, p_test)
})

plot_coefficients(model_raw.coef_.ravel(), X_full.columns, "Logistic Coefficients (Non-Standardized, X*)")


## 2) Logistic Regression with **Standardized** Features — Coefficient Plot

In [ ]:

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

RANDOM_STATE = 42
model_std = LogisticRegression(penalty=None, solver='lbfgs', max_iter=5000, random_state=RANDOM_STATE, class_weight='balanced')
model_std.fit(X_train_s, y_train)

p_test_s = model_std.predict_proba(X_test_s)[:, 1]
y_pred_s = (p_test_s >= 0.5).astype(int)

print("Test metrics (standardized features):")
print({
    'accuracy': accuracy_score(y_test, y_pred_s),
    'precision': precision_score(y_test, y_pred_s, zero_division=0),
    'recall': recall_score(y_test, y_pred_s, zero_division=0),
    'f1': f1_score(y_test, y_pred_s, zero_division=0),
    'roc_auc': roc_auc_score(y_test, p_test_s) if len(np.unique(y_test)) == 2 else np.nan,
    'log_loss': log_loss(y_test, p_test_s)
})

plot_coefficients(model_std.coef_.ravel(), X_full.columns, "Logistic Coefficients (Standardized, x-only)")


### 3) Forward Selection (Standardized) with **Undersampling** of Majority Class (X*)

In [ ]:

# Identify classes
mask_alive = (y_full == 0)
mask_failed = (y_full == 1)

X_alive = X_full[mask_alive].copy()
X_failed = X_full[mask_failed].copy()
y_alive = y_full[mask_alive].copy()
y_failed = y_full[mask_failed].copy()

n_minority = len(y_failed)
X_alive_down = resample(X_alive, replace=False, n_samples=n_minority, random_state=RANDOM_STATE)
y_alive_down = resample(y_alive, replace=False, n_samples=n_minority, random_state=RANDOM_STATE)

# Combine and shuffle
X_bal = pd.concat([X_alive_down, X_failed], axis=0)
y_bal = pd.concat([y_alive_down, y_failed], axis=0)
RANDOM_STATE = 42
perm = np.random.RandomState(RANDOM_STATE).permutation(len(y_bal))
X_bal = X_bal.iloc[perm].reset_index(drop=True)
y_bal = y_bal.iloc[perm].reset_index(drop=True)

print("Balanced shapes:", X_bal.shape, y_bal.shape)
print("Balanced class counts:", y_bal.value_counts().to_dict())

# Standardize balanced
scaler_bal = StandardScaler()
X_bal_std = scaler_bal.fit_transform(X_bal.values)
feat_bal = X_bal.columns.tolist()

sel_bal_aic, hist_bal_aic = forward_selection(X_bal_std, y_bal.values, feat_bal, criterion='AIC', max_steps=None, verbose=True)
print("\nSelected (balanced, AIC):", [feat_bal[i] for i in sel_bal_aic])
plot_history_metrics(hist_bal_aic, title_prefix="Forward Selection (Balanced AIC, x-only)")

sel_bal_bic, hist_bal_bic = forward_selection(X_bal_std, y_bal.values, feat_bal, criterion='BIC', max_steps=None, verbose=False)
print("\nSelected (balanced, BIC):", [feat_bal[i] for i in sel_bal_bic])
plot_history_metrics(hist_bal_bic, title_prefix="Forward Selection (Balanced BIC, x-only)")


In [ ]:
# === NEW: L1-penalized (LASSO) Logistic Regression — 4 C values ===
# Note: smaller C = stronger regularization (more sparsity)
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, log_loss
import numpy as np

# Reuse train/test if you set them earlier; otherwise fall back to full data
try:
    X_base, y_base = X_train, y_train
    X_eval, y_eval = X_test, y_test
except NameError:
    X_base, y_base = X_full, y_full
    X_eval, y_eval = X_full, y_full

scaler_l1 = StandardScaler()
X_base_std = scaler_l1.fit_transform(X_base)
X_eval_std = scaler_l1.transform(X_eval)

Cs = [0.001, 0.01, 0.1, 1.0]  # four values as requested
for C in Cs:
    l1 = LogisticRegression(
        penalty='l1',
        solver='liblinear',     # binary; 'saga' also works if you prefer
        C=C,
        max_iter=5000,
        random_state=RANDOM_STATE,
        class_weight='balanced'
    )
    l1.fit(X_base_std, y_base)
    p = l1.predict_proba(X_eval_std)[:, 1]
    y_hat = (p >= 0.5).astype(int)

    print(f"\nL1 Logistic (C={C}) — metrics:")
    print({
        'accuracy':  accuracy_score(y_eval, y_hat),
        'precision': precision_score(y_eval, y_hat, zero_division=0),
        'recall':    recall_score(y_eval, y_hat, zero_division=0),
        'f1':        f1_score(y_eval, y_hat, zero_division=0),
        'roc_auc':   roc_auc_score(y_eval, p) if len(np.unique(y_eval)) == 2 else np.nan,
        'log_loss':  log_loss(y_eval, p)
    })

    coef = l1.coef_.ravel()
    nz = int(np.count_nonzero(coef))
    print(f"Non-zero coefficients: {nz}/{coef.size}")

    # Same style coefficient plot as before
    plot_coefficients(coef, X_full.columns, f"L1 Logistic Coefficients (C={C}, standardized)")
